# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities, such as record sets, fields, and columns, are referenced by their `@id`.

In [ ]:
# List all available record sets and their fields by @id
print("Available Record Sets and Fields (all referenced by @id):\n")

record_sets = []

for record_set in metadata.record_sets:
    rs_id = record_set.id
    record_sets.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    for field in record_set.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print("")

if not record_sets:
    print("No record sets were found in the schema. Please check the dataset schema or contact the data provider.")
else:
    print(f"RecordSet @ids: {record_sets}")

Let's inspect a sample of records from each record set using their `@id`. (If you have a particular record set, replace in the code below.)

In [ ]:
# Print the first 3 records from each record set using its @id
# We use the first found record set as an example
example_record_set = None

if record_sets:
    example_record_set = record_sets[0]
    print(f"Showing the first 3 records for RecordSet @id: {example_record_set}")
    for idx, record in enumerate(dataset.records(record_set=example_record_set)):
        if idx >= 3:
            break
        print(json.dumps(record, indent=2))
else:
    print("No record sets available to preview records.")

## 3. Data Extraction
Load data from available record set(s) into a DataFrame for analysis. Reference all sets by their `@id`.

In [ ]:
# Extract data from all record sets into Pandas DataFrames using their @id
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records from RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"RecordSet {rs_id} DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("")

if not dataframes:
    print("No dataframes loaded. No record sets in the metadata.")
else:
    current_rs_id = record_sets[0]
    print(f"\nPreviewing columns for RecordSet @id: {current_rs_id}:")
    print(dataframes[current_rs_id].columns.tolist())
    dataframes[current_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**All entities are referenced by their `@id`.**

In [ ]:
# Choose a numeric field for EDA from the selected record set

# List all numeric fields (dataType Integer or Float) for the first record set
numeric_fields = []
group_candidate = None
if record_sets:
    first_rs = None
    for rs in metadata.record_sets:
        if rs.id == record_sets[0]:
            first_rs = rs
            break
    if first_rs is not None:
        for field in first_rs.fields:
            if field.data_type in ("schema:Integer", "schema:Float", "schema:Number"):
                numeric_fields.append(field.id)
            # Propose the first field of dataType 'schema:Text' for grouping, if available
            if not group_candidate and field.data_type == "schema:Text":
                group_candidate = field.id

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    numeric_field_id = None
    print("No numeric fields detected. Please specify one for your analysis.")

# Set variables for record set @id and group field @id (text field for grouping)
record_set_id = record_sets[0] if record_sets else None
group_field = group_candidate if group_candidate else None

# Proceed if a numeric field is present
if numeric_field_id and record_set_id:
    df = dataframes[record_set_id]

    # Convert the field to numeric if it's not
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If a group field is available, show grouped means
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean')
        print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("Numeric field or record set not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot a histogram of the selected numeric field, and, if available, a boxplot grouped by the text field referenced by its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and record_set_id:
    df = dataframes[record_set_id]
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group, if possible
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field and/or group field available for visualization.")

## 6. Conclusion
Summarized below are the key findings and observations from this exploration:
- Successfully loaded dataset metadata and inspected record sets and fields by their `@id`.
- Loaded records from each record set using `mlcroissant` (referenced by `@id`).
- Performed data filtering and normalization for a numeric field, and grouped records by a text field (if present).
- Visualized data distributions and relationships using basic plots.

This notebook demonstrates a reproducible, `mlcroissant`-powered data analysis workflow, referencing all entities by their schema `@id` for maximum clarity and FAIR practice.